In [1]:
"""This file contains code needed to prepare the scraped Epicurious recipe
JSON to convert to a database that can be used for cosine similarity analysis.
"""

# Import necessary libraries
import csv
import joblib
import json
import nltk

nltk.download("wordnet")
nltk.download("stopwords")
nltk.download("punkt")
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import numpy as np
import os
import pandas as pd
import re
import requests
import string
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from typing import Dict, Text

[nltk_data] Downloading package wordnet to /home/awchen/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /home/awchen/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/awchen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Load data needed for database preparation
stopwords_loc = "../data/food_stopwords.csv"
with open(stopwords_loc, "r") as myfile:
    reader = csv.reader(myfile)
    food_stopwords = [col for row in reader for col in row]

stopwords_list = stopwords.words("english") + list(string.punctuation) + food_stopwords

lemmatizer = WordNetLemmatizer()

In [3]:
# Define functions
def cuisine_namer(text):
    """This function converts redundant and/or rare categories into more common
    ones/umbrella ones.

    In the future, there's a hope that this renaming mechanism will not have
    under sampled cuisine tags.
    """
    if text == "Central American/Caribbean":
        return "Caribbean"
    elif text == "Jewish":
        return "Kosher"
    elif text == "Eastern European/Russian":
        return "Eastern European"
    elif text in ["Spanish/Portuguese", "Greek"]:
        return "Mediterranean"
    elif text == "Central/South American":
        return "Latin American"
    elif text == "Sushi":
        return "Japanese"
    elif text == "Southern Italian":
        return "Italian"
    elif text in ["Southern", "Tex-Mex"]:
        return "American"
    elif text in ["Southeast Asian", "Korean"]:
        return "Asian"
    else:
        return text


def link_maker(recipe_link: Text) -> Text:
    """This function takes in the incomplete recipe link from the dataframe and returns the complete one."""
    full_link = f"https://www.epicurious.com{recipe_link}"
    return full_link


def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """This function takes in a pandas DataFrame from pd.read_json and performs some preprocessing by unpacking the nested dictionaries and creating new columns with the simplified structures. It will then drop the original columns that would no longer be needed.

    Args:
        pd.DataFrame

    Returns:
        pd.DataFrame
    """

    def null_filler(to_check: Dict[str, str], key_target: str) -> str:
        """This function takes in a dictionary that is currently fed in with a lambda function and then performs column specific preprocessing.

        Args:
            to_check: dict
            key_target: str

        Returns:
            str
        """

        # Only look in the following keys, if the input isn't one of these, it should be recognized as an improper key
        valid_keys = ["name", "filename", "credit"]

        # This dictionary converts the input keys into substrings that can be used in f-strings to fill in missing values in the record
        translation_keys = {
            "name": "Cuisine",
            "filename": "Photo",
            "credit": "Photo Credit",
        }

        if key_target not in valid_keys:
            # this logic makes sure we are only looking at valid keys
            return (
                "Improper key target: can only pick from 'name', 'filename', 'credit'."
            )

        else:
            if pd.isna(to_check):
                # this logic checks to see if the dictionary exists at all. if so, return Missing
                return f"Missing {translation_keys[key_target]}"
            else:
                if key_target == "name" and (to_check["category"] != "cuisine"):
                    # This logic checks for the cuisine, if the cuisine is not there (and instead has 'ingredient', 'type', 'item', 'equipment', 'meal'), mark as missing
                    return f"Missing {translation_keys[key_target]}"
                else:
                    # Otherwise, there should be no issue with returning
                    return to_check[key_target]

    # Dive into the tag column and extract the cuisine label. Put into new column or fills with "missing data"
    df["cuisine_name"] = df["tag"].apply(
        lambda x: null_filler(to_check=x, key_target="name")
    )
    # df["cuisine_name"] = df["tag"].apply(lambda x: x['name'] if not pd.isna(x) and x['category'] == 'cuisine' else 'Cuisine Missing')

    # this lambda function goes into the photo data column and extracts just the filename from the dictionary
    df["photo_filename"] = df["photoData"].apply(
        lambda x: null_filler(to_check=x, key_target="filename")
    )
    # df["photo_filename"] = df['photoData'].apply(lambda x: x['filename'] if not pd.isna(x) else 'Missing photo')

    # This lambda function goes into the photo data column and extracts just the photo credit from the dictionary
    df["photo_credit"] = df["photoData"].apply(
        lambda x: null_filler(to_check=x, key_target="credit")
    )
    # df["photo_credit"] = df['photoData'].apply(lambda x: x['credit'] if not pd.isna(x) else 'Missing credit')

    # for the above, maybe they can be refactored to one function where the arguments are a column name, dictionary key name, the substring return

    # this lambda funciton goes into the author column and extract the author name or fills iwth "missing data"
    df["author_name"] = df["author"].apply(
        lambda x: x[0]["name"] if x else "Missing Author Name"
    )

    # This function takes in the given pubDate column and creates a new column with the pubDate values converted to datetime objects
    df["date_published"] = pd.to_datetime(df["pubDate"], infer_datetime_format=True)

    df["imputed_cuisine_name"] = df["cuisine_name"].apply(cuisine_namer)
    df["recipe_url"] = df["url"].apply(link_maker)

    # drop some original columns to clean up the dataframe
    df.drop(
        labels=[
            "tag",
            # "photoData",
            "author",
            "type",
            "dateCrawled",
            "pubDate",
            "url",
        ],
        axis=1,
        inplace=True,
    )

    return df


# def prep_data(X):
#     """This function takes a dataframe X, drops columns that will not be used,
#     expands the hierarchical column into the dataframe, renames the columns
#     to be more human-readable, and drops one column created during dataframe
#     expansion"""
#     X.drop(
#         [
#             "pubDate",
#             "author",
#             "type",
#             "aggregateRating",
#             "reviewsCount",
#             "willMakeAgainPct",
#             "dateCrawled",
#             "prepSteps",
#         ],
#         axis=1,
#         inplace=True,
#     )

#     X.rename({"url": "recipe_url"}, axis=1, inplace=True)

#     concat = pd.concat([X.drop(["tag"], axis=1), X["tag"].apply(pd.Series)], axis=1)
#     concat.drop(
#         [
#             0,
#             "photosBadgeAltText",
#             "photosBadgeFileName",
#             "photosBadgeID",
#             "photosBadgeRelatedUri",
#             "url",
#         ],
#         axis=1,
#         inplace=True,
#     )

#     cuisine_only = concat[concat["category"] == "cuisine"]
#     cuisine_only.dropna(axis=0, inplace=True)
#     cuisine_only["imputed_label"] = cuisine_only["name"].apply(cuisine_namer)
#     cuisine_only.drop("name", axis=1, inplace=True)
#     return cuisine_only


def fit_transform_tfidf_matrix(X_df, stopwords_list):
    tfidf = TfidfVectorizer(
        stop_words=stopwords_list,
        min_df=2,
        token_pattern=r"(?u)\b[a-zA-Z]{2,}\b",
        preprocessor=lemmatizer.lemmatize,
    )

    # temp = X_df["ingredients"].apply(" ".join).str.lower()
    temp = (
        X_df["ingredients"].str.join(" || ").fillna("Missing ingredients")
    )  # .tolist()#.apply(" ".join)
    tfidf.fit(temp)
    response = tfidf.transform(temp)
    word_matrix = pd.DataFrame(
        response.toarray(), columns=tfidf.get_feature_names_out(), index=X_df.index
    )

    return tfidf, word_matrix


def transform_tfidf(tfidf, recipe):
    ingreds = (
        recipe["ingredients"].str.join(" || ").fillna("Missing ingredients")
    )  # .apply(" ".join)#.str.lower()
    response = tfidf.transform(ingreds)

    transformed_recipe = pd.DataFrame(
        response.toarray(), columns=tfidf.get_feature_names_out(), index=recipe.index
    )
    return transformed_recipe


def transform_from_test_tfidf(tfidf, df, idx):
    recipe = (
        df["ingredients"].iloc[idx].str.join(" || ").fillna("Missing ingredients")
    )  # .apply(" ".join)#.str.lower()
    response = tfidf.transform(recipe)
    transformed_recipe = pd.DataFrame(
        response.toarray(), columns=tfidf.get_feature_names_out()
    )
    return transformed_recipe


def filter_out_cuisine(ingred_word_matrix, X_df, cuisine_name, tfidf):
    combo = pd.concat([ingred_word_matrix, X_df["imputed_cuisine_name"]], axis=1)
    filtered_ingred_word_matrix = combo[
        combo["imputed_cuisine_name"] != cuisine_name
    ].drop("imputed_cuisine_name", axis=1)
    return filtered_ingred_word_matrix


def find_closest_recipes(filtered_ingred_word_matrix, recipe_tfidf, X_df):
    search_vec = np.array(recipe_tfidf).reshape(1, -1)
    res_cos_sim = cosine_similarity(filtered_ingred_word_matrix, search_vec)
    top_five = np.argsort(res_cos_sim.flatten())[-5:][::-1]
    proximity = res_cos_sim[top_five]
    recipe_ids = [filtered_ingred_word_matrix.iloc[idx].name for idx in top_five]
    suggest_df = X_df.loc[recipe_ids]
    return suggest_df, proximity


def transform_tfidf(ingred_tfidf, recipe):
    # This function takes in a TFIDF Vectorizer object and a recipe, then
    # creates/transforms the given recipe into a TFIDF form

    ingreds = recipe["ingredients"].apply(" ".join).str.lower()
    response = ingred_tfidf.transform(ingreds)
    transformed_recipe = pd.DataFrame(
        response.toarray(),
        columns=ingred_tfidf.get_feature_names_out(),
        index=recipe.index,
    )
    return transformed_recipe


def filter_out_cuisine(ingred_word_matrix, X_df, cuisine_name, tfidf):
    # This function takes in the ingredient word matrix (from joblib), a
    # dataframe made from the database (from joblib), the user inputted cuisine
    # name, and the ingredient TFIDF Vectorizer object (from joblib) and returns
    # a word sub matrix that removes all recipes with the same cuisine as the
    # inputted recipe.

    east_asian = ["Asian", "Chinese", "Japanese"]

    southeast_asian = ["Asian", "Thai", "Vietnamese"]

    euro_islands = ["English", "Irish"]

    euro_continental = ["French", "German", "Eastern European"]

    mediterranean = ["Italian", "Mediterranean", "Kosher", "Middle Eastern"]

    all_cuisines = [
        "African",
        "American",
        "Asian",
        "Cajun/Creole",
        "Chinese",
        "Eastern European",
        "English",
        "French",
        "German",
        "Indian",
        "Irish",
        "Italian",
        "Japanese",
        "Kosher",
        "Latin American",
        "Mediterranean",
        "Mexican",
        "Middle Eastern",
        "Moroccan",
        "Scandinavian",
        "Southwestern",
        "Thai",
        "Vietnamese",
    ]

    if cuisine_name in east_asian:
        choices = [cuis for cuis in all_cuisines if cuis not in east_asian]
    elif cuisine_name in southeast_asian:
        choices = [cuis for cuis in all_cuisines if cuis not in southeast_asian]
    elif cuisine_name in euro_islands:
        choices = [cuis for cuis in all_cuisines if cuis not in euro_islands]
    elif cuisine_name in euro_continental:
        choices = [cuis for cuis in all_cuisines if cuis not in euro_continental]
    elif cuisine_name in mediterranean:
        choices = [cuis for cuis in all_cuisines if cuis not in mediterranean]
    else:
        choices = [cuis for cuis in all_cuisines if cuis != cuisine_name]

    combo = pd.concat([ingred_word_matrix, X_df["imputed_cuisine_name"]], axis=1)
    filtered_ingred_word_matrix = combo[
        combo["imputed_cuisine_name"].isin(choices)
    ].drop("imputed_cuisine_name", axis=1)

    return filtered_ingred_word_matrix


def picture_placer(filename):
    # This function takes in a filename and returns the relative location inside
    # an HTML tag
    location = f"photos/{filename}"
    return location


def link_maker(recipe_link):
    # This function takes in the incomplete recipe link from the dataframe and
    # returns the complete one.
    full_link = f"https://www.epicurious.com{recipe_link}"
    return full_link


def find_closest_recipes(filtered_ingred_word_matrix, recipe_tfidf, X_df):
    # This function takes in the filtered ingredient word matrix from function
    # filter_out_cuisine, the TFIDF recipe from function transform_tfidf, and
    # a dataframe made from the database (from joblib) and returns a Pandas
    # DataFrame with the top five most similar recipes and a Pandas Series
    # containing the similarity amount

    m2 = (recipe_tfidf != 0).any()
    recipe_weights = recipe_tfidf.iloc[0][recipe_tfidf.iloc[0] != 0].to_dict()

    # print(recipe_weights)
    print(
        recipe_tfidf.iloc[0][recipe_tfidf.iloc[0] != 0]
        .T.sort_values(ascending=False)
        .head()
        .T.to_dict()
    )

    ingreds_used = m2.index[m2].tolist()
    search_vec = np.array(recipe_tfidf).reshape(1, -1)
    res_cos_sim = cosine_similarity(filtered_ingred_word_matrix, search_vec)
    top_five = np.argsort(res_cos_sim.flatten())[-5:][::-1]
    top_five_list = top_five.tolist()

    recipe_ids = [filtered_ingred_word_matrix.iloc[idx].name for idx in top_five]

    suggest_df = X_df.loc[recipe_ids]
    proximity = pd.DataFrame(
        data=res_cos_sim[top_five],
        columns=["cosine_similarity"],
        index=suggest_df.index,
    )

    full_df = pd.concat([suggest_df, proximity], axis=1)
    expand_photo_df = pd.concat(
        [full_df.drop(["photoData"], axis=1), full_df["photoData"].apply(pd.Series)],
        axis=1,
    )
    reduced = expand_photo_df[
        [
            "hed",
            "recipe_url",
            "filename",
            "imputed_cuisine_name",
            "ingredients",
            "cosine_similarity",
        ]
    ].dropna(axis=1)
    reduced["photo"] = reduced["filename"].apply(picture_placer)
    reduced["fixed_url"] = reduced["recipe_url"].apply(link_maker)
    reduced["rounded"] = reduced["cosine_similarity"].round(3)

    reduced = reduced.drop("recipe_url", axis=1)

    ingr_weights = [
        filtered_ingred_word_matrix.iloc[num][
            filtered_ingred_word_matrix.iloc[num] != 0
        ].to_dict()
        for num in top_five_list
    ]
    reduced["ingred_weights"] = ingr_weights

    return reduced, ingreds_used, recipe_weights


def find_similar_dishes(dish_name, cuisine_name):
    # epic_dataframe, ingred_tfidf, ingred_word_matrix = import_stored_files()
    # This function calls the Edamam API, stores the results as a JSON, and
    # stores the timestamp, dish name, and cuisine name/classification in a
    # separate csv.

    api_base = "https://api.edamam.com/api/recipes/v2?type=public&"

    # Level up:
    # Check a database of dishes to see if this query has been asked for already
    # If not, do an API call

    # Currently, just does an API call, may hit API limit if continuing with this version
    # cred_appid = os.environ["EDAMAM_API_APPID"]
    # cred_appkey = os.environ["EDAMAM_API_APPKEY"]
    with open("../secrets/edamam_api.json", "r") as f:
        creds = json.load(f)
        cred_appid = creds["EDAMAM_API_APPID"]
        cred_appkey = creds["EDAMAM_API_APPKEY"]

        api_call = f"{api_base}q={dish_name}&app_id={cred_appid}&app_key={cred_appkey}"

        resp = requests.get(api_call)

        if resp.status_code == 200:
            response_dict = resp.json()
            resp_dict_hits = response_dict["hits"]

            # Store the API result into a JSON and the cuisine type and dish name into a
            # csv
            # Heroku does not save files to directory
            # Can work with EC2
            # with open(f"../write_data/{dt_string}_{dish_name}_edamam_api_return.json", "w") as f:
            #   json.dump(resp_dict_hits, f)

            urls = []
            labels = []
            sources = []
            ingreds = []
            cuisines = []

            for recipe in resp_dict_hits:
                recipe_path = recipe["recipe"]
                urls.append(recipe_path["url"])
                labels.append(recipe_path["label"])
                sources.append(recipe_path["source"])
                ingreds.append(recipe_path["ingredientLines"])
                cuisines.append(recipe_path["cuisineType"])

            all_recipes = {
                "url": urls,
                "label": labels,
                "source": sources,
                "ingredients": ingreds,
                "cuisines": cuisines,
            }

            one_recipe = []

            for listing in all_recipes["ingredients"]:
                for ingred in listing:
                    one_recipe.append(ingred.lower())

            one_recipe = list(set(one_recipe))

            query_df = pd.DataFrame(
                data={
                    "name": dish_name,
                    "ingredients": [one_recipe],
                    "cuisine": cuisine_name,
                }
            )

            query_tfidf = transform_tfidf(ingred_tfidf=ingred_tfidf, recipe=query_df)

            query_matrix = filter_out_cuisine(
                ingred_word_matrix=ingred_word_matrix,
                X_df=epic_dataframe,
                cuisine_name=cuisine_name,
                tfidf=ingred_tfidf,
            )

            query_similar, ingreds_used, recipe_weights = find_closest_recipes(
                filtered_ingred_word_matrix=query_matrix,
                recipe_tfidf=query_tfidf,
                X_df=epic_dataframe,
            )

            return query_similar.to_dict(orient="records"), ingreds_used, recipe_weights

        else:
            print(
                f"Error, unable to retrieve. Server response code is: {resp.status_code}"
            )
            return ([[]], [[]], [[]])

In [4]:
# Create the dataframe
epic_dataframe = pd.read_json(
    "../data/recipes-en-201706/epicurious-recipes_m2.json", typ="frame"
)

preprocess_dataframe(df=epic_dataframe)
epic_dataframe.head(3)

,id,dek,hed,photoData,aggregateRating,ingredients,prepSteps,reviewsCount,willMakeAgainPct,cuisine_name,photo_filename,photo_credit,author_name,date_published,imputed_cuisine_name,recipe_url
0,54a2b6b019925f464b373351,How does fried chicken achieve No. 1 status? B...,Pickle-Brined Fried Chicken,"{'id': '54a2b64a6529d92b2c003409', 'filename':...",3.11,"[1 tablespoons yellow mustard seeds, 1 tablesp...",[Toast mustard and coriander seeds in a dry me...,7,100,Missing Cuisine,51247610_fried-chicken_1x1.jpg,Michael Graydon and Nikole Herriott,Missing Author Name,2014-08-19 04:00:00+00:00,Missing Cuisine,https://www.epicurious.com/recipes/food/views/...
1,54a408a019925f464b3733bc,Spinaci all'Ebraica,Spinach Jewish Style,"{'id': '56746182accb4c9831e45e0a', 'filename':...",3.22,"[3 pounds small-leaved bulk spinach, Salt, 1/2...",[Remove the stems and roots from the spinach. ...,5,80,Italian,EP_12162015_placeholders_rustic.jpg,"Photo by Chelsea Kyle, Prop Styling by Anna St...",Edda Servi Machlin,2008-09-09 04:00:00+00:00,Italian,https://www.epicurious.com/recipes/food/views/...
2,54a408a26529d92b2c003631,"This majestic, moist, and richly spiced honey ...",New Year’s Honey Cake,"{'id': '55e85ba4cf90d6663f728014', 'filename':...",3.62,"[3 1/2 cups all-purpose flour, 1 tablespoon ba...",[I like this cake best baked in a 9-inch angel...,105,88,Jewish,EP_09022015_honeycake-2.jpg,"Photo by Chelsea Kyle, Food Styling by Anna St...",Marcy Goldman,2008-09-10 04:00:00+00:00,Kosher,https://www.epicurious.com/recipes/food/views/...


In [5]:
ingred_tfidf, ingred_word_matrix = fit_transform_tfidf_matrix(
    epic_dataframe, stopwords_list
)

/home/awchen/Repos/Projects/MeaLeon/.venv/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['Frank', 'alternatives', 'annie', 'balance', 'band', 'barrel', 'bayou', 'beam', 'beard', 'bell', 'betty', 'bird', 'blast', 'bob', 'bone', 'breyers', 'calore', 'carb', 'card', 'change', 'circle', 'clove', 'coffee', 'coil', 'country', 'cow', 'crack', 'cracker', 'crocker', 'crystal', 'dean', 'degree', 'deluxe', 'direction', 'doe', 'drip', 'duncan', 'earth', 'eggland', 'ener', 'envelope', 'eye', 'fantastic', 'fat', 'feather', 'flake', 'foot', 'fourth', 'fragment', 'frank', 'fry', 'fusion', 'genovese', 'germain', 'giada', 'gold', 'grands', 'granule', 'hamburger', 'heart', 'helper', 'hidden', 'hines', 'hodgson', 'hunt', 'instruction', 'interval', 'jim', 'jimmy', 'kellogg', 'lagrille', 'lakes', 'land', 'laughing', 'laurentiis', 'lawry', 'leaf', 'lipton', 'litre', 'mai

In [6]:
find_similar_dishes("lasagna", "Italian")

{'lasagna': 0.7371297923338641, 'noodles': 0.2603845399505066, 'cheese': 0.2566690143549585, 'ricotta': 0.19394866106969197, 'mozzarella': 0.19084476785463195}


([{'hed': 'Lean Lasagna',
   'filename': 'lean-lasagna.jpg',
   'imputed_cuisine_name': 'American',
   'ingredients': ['Vegetable-oil cooking spray',
    '1/2 cup chopped onion',
    '1 lb ground turkey breast',
    '3 cups tomato sauce',
    '3 tsp Italian seasoning (or 1 tsp each dried basil, parsley, and oregano)',
    '1/4 tsp freshly ground black pepper',
    '1/4 tsp garlic powder',
    '1/2 cup chopped mushrooms',
    '6 cups chopped fresh spinach (or chard)',
    '2 cups fat-free ricotta',
    '1/4 tsp nutmeg',
    '1 package whole-wheat lasagna noodles(about 8 oz, or 9 noodles)',
    '2 cups (8 oz) shredded part-skim mozzarella'],
   'cosine_similarity': 0.5825168147272504,
   'photo': 'photos/lean-lasagna.jpg',
   'fixed_url': 'https://www.epicurious.comhttps://www.epicurious.com/recipes/food/views/lean-lasagna-230145',
   'rounded': 0.583,
   'ingred_weights': {'Vegetable': 0.2202291486446679,
    'basil': 0.1696630151682376,
    'black': 0.11266830989083243,
    'chard': 0.

In [ ]:
recipe_tfidf

In [ ]:
# save the preprocessed dataframe
with open("../joblib/prepped_dataframe.joblib", "wb") as fo:
    joblib.dump(epic_dataframe, fo, compress=True)

# Create TF-IDF transformer (ingred_tfidf) and word matrix (recipe_word_matrix_tfidf)

with open("../joblib/recipe_tfidf.joblib", "wb") as fo:
    joblib.dump(ingred_tfidf, fo, compress=True)

with open("../joblib/recipe_word_matrix_tfidf.joblib", "wb") as fo:
    joblib.dump(ingred_word_matrix, fo, compress=True)